# Week 10 Live Coding
## Testing a turnout model

Keystone Analytics sold us a turnout score for all 60,000 registered voters in the district. We have the same voter file they used, so we can test it.

Four things we will do:
1. Test the file Keystone sent, starting with the accuracy floor
2. Build our own model and test it on voters it has never seen
3. Check calibration, twice: our model, and Keystone's
4. Score everyone for 2026 and build the walk list

*(The file is synthetic. It was built for this class so that every number on the slides is exactly reproducible.)*

### Before you start

**Save your own copy first.** Go to **File → Save a copy in Drive**. A new tab opens with your own copy. Work in that tab; edits to the original are not saved.

**The data loads itself.** There is nothing to download or upload. The setup cell below pulls the voter file straight from the course repository; you just need to be online.

Stuck? See the Colab troubleshooting guide on the syllabus.

## Setup

Run the cell below. The file is 60,000 rows, so give it a couple of seconds.

In [ ]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
import matplotlib.pyplot as plt

vf = pd.read_csv('https://raw.githubusercontent.com/joshuakalla/data_science_campaigns_26/'
                 'main/weeks/wk10_turnout_models/data/district_voter_file.csv')
print(vf.shape)
vf.head()

## Part 1: What did we buy?

One row per registered voter. Seven columns of vote history, three of demographics, and `keystone_score`.

Start with what actually happened: the share who voted in each election.

In [ ]:
# Turnout on this file, election by election
for year in [2012, 2014, 2016, 2018, 2020, 2022, 2024]:
    print(year, round(100 * vf['voted_' + str(year)].mean(), 1), '%')

Presidential years run from 57% to 76%. Midterms run from 41% to 51%. More people vote in presidential years.

### Holding voters back

We are about to build our own model. If we test it on the same voters we built it from, it can look good by memorizing them. So set aside a random 18,000 voters now, and run every test on them.

(We cannot do this for Keystone. Their model has already seen all 60,000 voters.)

In [ ]:
train = vf.sample(frac=0.7, random_state=10)
test = vf.drop(train.index).copy()
print(len(train), 'to build on;', len(test), 'held back')

### The accuracy floor

Keystone's claim: *validated against the 2024 general election, the model classified 80.0% of registered voters correctly.*

Call anyone scored 0.5 or above a voter and anyone below a non-voter, and compare to who voted in 2024.

Before you judge 80.0%, compare it to a model that writes the more common answer on every row. Call its accuracy the **free answer**.

In [ ]:
keystone_call = (test['keystone_score'] >= 0.5).astype(int)
accuracy = (keystone_call == test['voted_2024']).mean()

turnout_2024 = test['voted_2024'].mean()
free_2024 = max(turnout_2024, 1 - turnout_2024)   # the more common answer

print('Keystone accuracy:', round(100 * accuracy, 1), '%')
print('The free answer:  ', round(100 * free_2024, 1), '%')
print('Above the free answer:', round(100 * (accuracy - free_2024), 1), 'points')

Keystone's model is 4.1 points above the free answer. Most of its 80.0% comes from the share of people who vote in a presidential year.

## Part 2: Build our own

**Step 1: pick an election.** Our race is a 2026 midterm, so predict 2022, the last midterm.

**Step 2: find the pattern.** Regress 2022 turnout on the four general elections before 2020, plus age. Fit it on the 42,000 training voters only.

In [ ]:
X = 'voted_2012 + voted_2014 + voted_2016 + voted_2018 + age'
mid = smf.ols('voted_2022 ~ ' + X, data=train).fit()

(100 * mid.params).round(1)

Multiply by 100 and a coefficient reads in percentage points, same as Week 2.

**2014 and 2018 were midterms. 2012 and 2016 were presidentials.** To predict a midterm, having voted in a past midterm is worth 27.7 and 33.1 points. Having voted in a past presidential is worth 10.4 and 11.8.

### A linear probability model

We fit a straight line to a 0/1 outcome. That is a **linear probability model**, and it is why we can read the coefficients in points. A straight line can go below 0 or above 1.

In [ ]:
test['our_score'] = mid.predict(test)
print('our scores run from', round(test['our_score'].min(), 2),
      'to', round(test['our_score'].max(), 2))

# The same model, fit to 2024 instead
pres = smf.ols('voted_2024 ~ ' + X, data=train).fit()
pres_score = pres.predict(test)
print('fit to 2024:', round(100 * (pres_score > 1).mean(), 1),
      '% of scores are above 1.00')

Midterm turnout is near 50%, so our scores stay between 0 and 1. In a presidential year they do not.

The fix is to **trim scores at 0 and 1**. Keystone's file comes trimmed, which is why 9,595 people are tied at exactly 1.00.

### Testing our model

Score the 18,000 held-out voters and compare with who voted in 2022.

In [ ]:
our_call = (test['our_score'] >= 0.5).astype(int)
accuracy = (our_call == test['voted_2022']).mean()

turnout_2022 = test['voted_2022'].mean()
free_2022 = max(turnout_2022, 1 - turnout_2022)

print('Our accuracy:', round(100 * accuracy, 1), '%')
print('The free answer:', round(100 * free_2022, 1), '%')
print('Above the free answer:', round(100 * (accuracy - free_2022), 1), 'points')

Our model is right about as often as Keystone's, but it is 30.4 points above its free answer, and Keystone's is 4.1.

### The confusion matrix

A **false positive**: the model said "will vote," and she did not. A **false negative**: the model said "will not vote," and she did.

In [ ]:
confusion = pd.crosstab(our_call, test['voted_2022'])
confusion.index = ['model said will not vote', 'model said will vote']
confusion.columns = ['did not vote', 'voted']
confusion

In [ ]:
tp = ((our_call == 1) & (test['voted_2022'] == 1)).sum()
fp = ((our_call == 1) & (test['voted_2022'] == 0)).sum()
fn = ((our_call == 0) & (test['voted_2022'] == 1)).sum()

print('precision:', round(100 * tp / (tp + fp), 1), '%   of those we called voters, the share who voted')
print('recall:   ', round(100 * tp / (tp + fn), 1), '%   of the people who voted, the share we called voters')
print()
print('assume everyone votes: precision', round(100 * turnout_2022, 1), '%, recall 100.0 %')

## Part 3: Calibration

Accuracy, precision and recall turn each score into a yes or a no. Calibration checks the score itself.

**A model is calibrated when its scores mean what they say: of the people it scores 0.60, about 60% vote.**

The helper below sorts voters into ranges of 10 points (0 to 0.1, 0.1 to 0.2, and so on). For each range it reports how many people are in it, their average score, and the share who actually voted.

In [ ]:
def calibration(scores, actual):
    # The last range has to reach the highest score, or pd.cut
    # leaves those people out without telling you.
    top = max(1.0, np.ceil(scores.max() * 10) / 10)
    bins = pd.cut(scores, np.arange(0, top + 0.001, 0.1), include_lowest=True)
    out = pd.DataFrame({'score': scores, 'voted': actual}).groupby(bins, observed=True).agg(
        n=('voted', 'size'), model_said=('score', 'mean'), actually_voted=('voted', 'mean'))
    out['gap_pts'] = (out['model_said'] - out['actually_voted']) * 100
    return out.round(3)

calibration(test['our_score'], test['voted_2022'])

In every range, the model is within about 4 points of the share who voted. The worst is the bottom range: the model said 8.5%, and 4.2% voted.

Now Keystone's scores. Keystone fit its model to the **2024 general**, a presidential year. Our race is a **2026 midterm**. You cannot check a 2026 score until after the election, but 2022 was a midterm and we know who voted. So test Keystone's scores on 2022.

In [ ]:
print('mean Keystone score:', round(100 * test['keystone_score'].mean(), 1), '%')
print('voted in 2022:      ', round(100 * test['voted_2022'].mean(), 1), '%')
print('Keystone accuracy on 2022:',
      round(100 * (keystone_call == test['voted_2022']).mean(), 1), '%')
calibration(test['keystone_score'], test['voted_2022'])

Keystone's scores average 75.6%, about the 2024 turnout. In every range, fewer people voted than Keystone's scores said.

The two side by side:

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4.4), sharey=True)

for ax, col, title, color in [
        (axes[0], 'our_score', 'Our model, tested on 2022', '#0F4D92'),
        (axes[1], 'keystone_score', "Keystone's scores, tested on 2022", '#B58900')]:
    tab = calibration(test[col], test['voted_2022'])
    ax.plot([0, 1], [0, 1], color='#555555', ls='--', lw=1)
    ax.scatter(tab['model_said'], tab['actually_voted'], s=60, color=color)
    ax.set_title(title)
    ax.set_xlabel("Voters' scores")
    ax.set_xlim(0, 1.05); ax.set_ylim(0, 1.05)

axes[0].set_ylabel('Share who actually voted in 2022')
plt.tight_layout()
plt.show()

## Part 4: The walk list

### Step 3: score everyone for 2026

Same coefficients. Move every voter's history forward four years: 2016 goes where 2012 was, 2018 where 2014 was, 2020 where 2016 was, and 2022 where 2018 was. Age goes up by 4.

In [ ]:
moved_forward = pd.DataFrame({'voted_2012': vf['voted_2016'],
                              'voted_2014': vf['voted_2018'],
                              'voted_2016': vf['voted_2020'],
                              'voted_2018': vf['voted_2022'],
                              'age': vf['age'] + 4})
vf['score_2026'] = mid.predict(moved_forward)

print('2026 scores run from', round(vf['score_2026'].min(), 2),
      'to', round(vf['score_2026'].max(), 2))

Sort all 60,000 into three ranges of 2026 score, and look at the three lists our office is arguing about. 9,000 doors each.

In [ ]:
vf['band'] = pd.cut(vf['score_2026'], [-1, 0.35, 0.65, 2],
                    labels=['low', 'middle', 'high'])
print(vf['band'].value_counts())
print()

middle_sample = vf[vf['band'] == 'middle'].sample(9000, random_state=3)
for name, rows in [('Rachel: 9,000 highest', vf.nlargest(9000, 'score_2026')),
                   ('Katie: 9,000 in the middle', middle_sample),
                   ('Marcus: 9,000 lowest', vf.nsmallest(9000, 'score_2026'))]:
    print(name.ljust(28), 'average 2026 score:',
          round(100 * rows['score_2026'].mean(), 1), '%')

The average score is the share of each list the model expects to vote if you do nothing: about 93% of Rachel's list, 55% of Katie's, and 10% of Marcus's.

### Whose scores?

The case defines Katie's list with Keystone's scores: everyone between 0.35 and 0.65. Compare that with the same range on our 2026 scores.

In [ ]:
katie_keystone = vf[vf['keystone_score'].between(0.35, 0.65)]
katie_ours = vf[vf['score_2026'].between(0.35, 0.65)]

for name, rows in [('by Keystone', katie_keystone), ('by our 2026 score', katie_ours)]:
    print(name.ljust(18), len(rows), 'people |',
          round(100 * rows['voted_2022'].mean(), 1), '% voted in 2022 |',
          'our 2026 score averages', round(100 * rows['score_2026'].mean(), 1), '%')
print('in both lists:', len(set(katie_keystone.index) & set(katie_ours.index)))

Keystone's "middle" is 22,482 people, and 14.0% of them voted in the last midterm. Ours is 10,814 people. The two lists share 3,141 people.

Rachel's and Marcus's lists only need the scores in the right order. Katie's list needs the scores themselves to be right, because it is defined by a range of scores.

### One more constraint

This is a get-out-the-vote program for a Democrat. Reminding a Republican to vote does not help you.

In [ ]:
counts = pd.crosstab(vf['band'], vf['party'])
print(counts)
print()
middle_d_or_u = counts.loc['middle', 'D'] + counts.loc['middle', 'U']
print('middle band, registered D or unaffiliated:', middle_d_or_u)
print('doors we can knock:', 9000)

Once you drop the Republicans, Katie's middle band has fewer than 9,000 people. You would have to widen it.

And party **registration** is not support: 30.8% of this district is unaffiliated, and a turnout model tells you nothing about who anyone supports.

Every score in this notebook is the probability someone votes **if you do nothing**. What you need is how much your knock **changes** that. That is a treatment effect, and nothing in the voter file contains it.